In [ ]:
# ====================== GPU PICKER (server-friendly) ======================
# Set which GPU to use (0, 1, ...). Set to None to NOT force anything (respects the environment).
GPU_ID = 1 # Change to None when you want to leave it free for other users
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import torch
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass
torch.backends.cudnn.benchmark = True  # if the size varies A LOT, consider False
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.set_device(0)
print("Device:", device, "| Visible devices (after mapping):", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if device == "cuda":
    try:
        print("Using mapped GPU 0:", torch.cuda.get_device_name(0))
    except Exception:
        pass
    for i in range(torch.cuda.device_count()):
        try:
            print(f"Device {i}:", torch.cuda.get_device_name(i))
        except Exception:
            pass
import sys
from datetime import datetime

In [ ]:
sys.path.insert(0, "/workspace/app")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from coding.Paper_CBMS import Run_RNN_dropout as rf
from sklearn.metrics import mean_absolute_error, mean_squared_error, recall_score, precision_score, f1_score, average_precision_score, precision_recall_fscore_support, roc_auc_score

____

Dataset

In [ ]:
df = pd.read_excel("/workspace/app/planilhas/CBMS/df_all_Horizon2.xlsx")

In [ ]:
df = df[df["Embedding_npy"].notna() & df["Embedding_npy"].apply(lambda x: isinstance(x, str))].copy()
df = df[df["Patient_ID"] != 2].copy()

In [ ]:
cols_remove = ['Age', "Patient_Gender", 'Education', 'Y_Zscore_TO_TOT_Alexithymia', 'Questionnary5_HQ_25_T0_soc_Score', 
              'Questionnary5_HQ_25_T0_iso_Score', 'Questionnary5_HQ_25_T0_sup_Score', 'Mean_Gap_Days', 'Std_Gap_Days', 
              'Y_Depression_T0', 'Embedding_npy_list']
df = df.drop(columns=cols_remove, errors="ignore")

In [ ]:
mfcc_mean = [f"MFCC{s}_mean" for s in range(1, 14)]

___

Datasets

* Experiment 1: Speech-only early warning 
1. Baseline MLP with MFFC + FO
2. GRU with wav2vec
3. GRU with wav2vec +  MFFC + FO (improved perfomance)

Complementary acoustic representations improve early-warning stability.”

In [ ]:
df_wav = df.copy()
df_wav = df_wav.drop(columns=mfcc_mean, errors="ignore")     
df_wav = df_wav.drop(columns=["F0_mean","F0_std"], errors="ignore")

In [ ]:
df_wav_late = df.copy()
tabular = mfcc_mean + ["F0_mean","F0_std"]

In [ ]:
datasets = [
    (df_wav, 'y_h2', None),
    (df_wav_late, 'y_h2', tabular)
]

GRU

In [ ]:
summary_results = []
all_results = []
BS = 32
epochs = 8
calibrate_tau="fixed"
warn_rule="fixed"
for idx, (df, target_column, tab_cols) in enumerate(datasets):
    print(f"\nProcessing Dataset {idx+1} - Target: {target_column}")
    unique_patients = df['Patient_ID'].unique()
    target_column = "y_h2"  
    results_gru = [rf.class_lopo_RNN_dropout(p, df, 'Patient_ID', target_column, "GRU", tab_cols = tab_cols, BS=BS, epochs=epochs, calibrate_tau=calibrate_tau, warn_rule=warn_rule)
        for p in unique_patients]
    all_results.extend(results_gru)
    df_current = pd.DataFrame(results_gru)
    df_current["Accuracy"] = np.where(
        df_current["y_patient_true"] == 1,
        (df_current["y_patient_pred"] == 1).astype(float),
        (df_current["y_patient_pred"] == 0).astype(float))
    df_current["BalAcc"] = (df_current["REC"].fillna(0) + df_current["SPEC"].fillna(0)) / 2
    summary_results.append({
                "Dataset_Index": idx + 1,
                "Accuracy_Macro_Mean": df_current["Accuracy"].mean(),
                "Accuracy_Macro_Std":  df_current["Accuracy"].std(),
                "BalAcc_Mean":          df_current["BalAcc"].mean(),
                "BalAcc_Std":           df_current["BalAcc"].std(),
                "Recall_Pos_Mean":     df_current.loc[df_current["y_patient_true"] == 1, "REC"].mean(),
                "Spec_Pos_Mean":       df_current["SPEC"].mean(), 
                "Precision_Pos_Mean":  df_current["PREC"].mean(),  
                "F1_Pos_Mean":         df_current["F1"].mean()})
df_results = pd.DataFrame(all_results)
df_summary = pd.DataFrame(summary_results)

In [ ]:
df_results.to_excel('/workspace/app/planilhas/CBMS/Exp1_GRU.xlsx', index=False)
df_summary.to_excel('/workspace/app/planilhas/CBMS/Exp1_GRU_SUMMARY.xlsx', index=False)